In [12]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from pydantic import BaseModel
from langchain.tools import tool
from pydantic import BaseModel, Field
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch, tavily_crawl


load_dotenv(find_dotenv())

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")



In [32]:
model = init_chat_model(
    model="qwen3.5-35b-a3b",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    # 尝试显式关闭思考模式
    extra_body={"enable_thinking": False} 
)

In [30]:
tavily = TavilySearch(
    max_results=1,
    topic="general"
)

@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

In [33]:
# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")
    
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="1+1等于几")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='1+1等于2。' reference=[]
